
# Allen–Cahn: traduzione MATLAB → Python

Questo notebook è una traduzione fedele dello script MATLAB fornito, usando:

- **NumPy** per algebra vettoriale e SVD;
- **SciPy sparse** per Laplaciano, Jacobiani e solve lineari sparsi;
- **pandas** per le tabelle;
- **Matplotlib** per i grafici.

### Convenzioni preservate

Per mantenere la corrispondenza numerica con MATLAB:

- i `reshape` che dipendono dall'ordinamento MATLAB usano `order="F"`;
- i parametri prodotti da `meshgrid` sono linearizzati in ordine Fortran (`order="F"`);
- la divisione matriciale destra MATLAB `A / B` è implementata senza formare esplicitamente `inv(B)`;
- gli indici DEIM sono convertiti correttamente da base 1 MATLAB a base 0 Python;
- l'algoritmo, i passi temporali, le basi ridotte, le maschere di Operator Inference e i criteri di selezione della regolarizzazione sono mantenuti invariati.

> Nota strutturale: nel file MATLAB le funzioni locali sono in fondo allo script. In Jupyter vengono definite prima del workflow, così le celle successive possono richiamarle direttamente.


In [ ]:

import time
import warnings

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import scipy.sparse as sp
import scipy.sparse.linalg as spla

from IPython.display import display

np.set_printoptions(precision=6, suppress=True)


## Funzioni locali

In [ ]:

def solve_newton(x0, F_fun, J_fun):
    """Metodo di Newton equivalente alla funzione MATLAB solve_newton."""
    tol = 1e-5
    err = 1.0
    iteration = 0
    max_iter = 100
    x_new = np.asarray(x0, dtype=float).reshape(-1).copy()

    while err > tol and iteration < max_iter:
        F_val = np.asarray(F_fun(x_new)).reshape(-1)
        J_val = J_fun(x_new)
        # MATLAB: delta_x = J_val \ F_val
        delta_x = spla.spsolve(J_val, F_val)
        x_new = x_new - delta_x
        err = np.linalg.norm(delta_x, ord=np.inf)
        iteration += 1

    if iteration == max_iter:
        warnings.warn('Newton non converge!', RuntimeWarning)

    return x_new


def simulate_allen_fom_implicit(L, y0, dt, times, save, snapshot_times, alpha, mu):
    """FOM Allen–Cahn con Eulero implicito e Newton, come nel MATLAB originale."""
    N = len(y0)
    state = np.asarray(y0, dtype=float).reshape(-1).copy()

    snapshot_matrix = np.zeros((N, len(snapshot_times)), dtype=float)
    snapshot_matrix[:, 0] = state

    I = sp.eye(N, format='csc')
    # Parte costante dello Jacobiano
    J_const = (I - dt * alpha * L - dt * mu * I).tocsc()

    save_counter = 0
    tic = time.perf_counter()

    for step in range(1, len(times)):
        def F_fun(x):
            return J_const @ x + dt * mu * (x ** 3) - state

        def J_fun(x):
            return J_const + sp.diags(3.0 * dt * mu * (x ** 2), offsets=0, format='csc')

        state = solve_newton(state, F_fun, J_fun)

        if step % save == 0:
            save_counter += 1
            snapshot_matrix[:, save_counter] = state

    runtime = time.perf_counter() - tic
    return snapshot_matrix[:, :save_counter + 1], runtime


def qvec(x):
    """Monomi quadratici xi*xj senza ripetizioni, con i <= j."""
    x = np.asarray(x).reshape(-1)
    r = len(x)
    q = np.zeros(r * (r + 1) // 2, dtype=float)

    row = 0
    for i in range(r):
        for j in range(i, r):
            q[row] = x[i] * x[j]
            row += 1

    return q


def qmat(X):
    """Applica qvec colonna per colonna: shape r(r+1)/2 x K."""
    X = np.asarray(X)
    r, K = X.shape
    Q = np.zeros((r * (r + 1) // 2, K), dtype=float)

    row = 0
    for i in range(r):
        for j in range(i, r):
            Q[row, :] = X[i, :] * X[j, :]
            row += 1

    return Q


def deim_indices(U):
    """Algoritmo greedy DEIM per la selezione dei punti spaziali."""
    U = np.asarray(U)
    indices = [int(np.argmax(np.abs(U[:, 0])))]

    for i in range(1, U.shape[1]):
        # MATLAB: c = U(indices, 1:i-1) \ U(indices, i)
        selected_matrix = U[np.ix_(indices, np.arange(i))]
        selected_rhs = U[indices, i]
        c = np.linalg.solve(selected_matrix, selected_rhs)

        residual = U[:, i] - U[:, :i] @ c
        new_index = int(np.argmax(np.abs(residual)))
        indices.append(new_index)

    return np.asarray(indices, dtype=int)


def train_allen_pod(L, snapshots, r, m):
    """Costruzione basi POD lineari e POD-DEIM."""
    # Basi POD lineari: MATLAB svd(..., 'econ')
    U, sv_linear, _ = np.linalg.svd(snapshots, full_matrices=False)
    Psi = U[:, :min(r, U.shape[1])]
    Lr = np.asarray(Psi.T @ (L @ Psi))

    # Basi POD non lineari
    Un, sv_nonlinear, _ = np.linalg.svd(snapshots ** 3, full_matrices=False)
    Un = Un[:, :min(m, Un.shape[1])]

    # Selezione DEIM
    deim_idx = deim_indices(Un)

    # MATLAB: nonlinear_projection = (Psi' * Un) / Un(deim_idx, :)
    # A / B = A @ inv(B); usiamo solve per maggiore stabilità numerica.
    A = Psi.T @ Un
    B = Un[deim_idx, :]
    nonlinear_projection = np.linalg.solve(B.T, A.T).T

    sampled_Psi = Psi[deim_idx, :]

    return (
        Psi,
        Lr,
        sv_linear,
        nonlinear_projection,
        sampled_Psi,
        deim_idx,
        sv_nonlinear,
    )


def build_allen_lift_data(L, alphas, mus, Y_all, num_snaps_per_sim, ry, rz):
    """Costruisce basi e dati per Lift & Learn / Operator Inference."""
    # LIFTING: T(y) = y.^2
    Z_all = Y_all ** 2

    # POD separata per stato originale e stato sollevato
    Uy, sv_y, _ = np.linalg.svd(Y_all, full_matrices=False)
    Psi_y = Uy[:, :ry]

    Uz, sv_z, _ = np.linalg.svd(Z_all, full_matrices=False)
    Psi_z = Uz[:, :rz]

    K_tot = Y_all.shape[1]
    X = np.zeros((ry + rz, K_tot), dtype=float)
    X_dot = np.zeros((ry + rz, K_tot), dtype=float)
    alpha_row = np.zeros(K_tot, dtype=float)
    mu_row = np.zeros(K_tot, dtype=float)

    for k in range(len(alphas)):
        a_k = alphas[k]
        m_k = mus[k]

        col_start = k * num_snaps_per_sim
        col_end = (k + 1) * num_snaps_per_sim
        states = Y_all[:, col_start:col_end]

        for j in range(num_snaps_per_sim):
            idx = col_start + j

            # Ricostruzione sullo spazio pieno filtrata dalla POD
            proj_y = Psi_y @ (Psi_y.T @ states[:, j])
            dy = a_k * (L @ proj_y) + m_k * (proj_y - proj_y ** 3)
            dz = 2.0 * proj_y * dy

            X[:, idx] = np.concatenate((
                Psi_y.T @ states[:, j],
                Psi_z.T @ (states[:, j] ** 2),
            ))

            X_dot[:, idx] = np.concatenate((
                Psi_y.T @ dy,
                Psi_z.T @ dz,
            ))

            alpha_row[idx] = a_k
            mu_row[idx] = m_k

    Q = qmat(X)

    # Broadcasting riga-per-riga equivalente a MATLAB X .* alpha_row
    alpha_lin = X * alpha_row[None, :]
    mu_lin = X * mu_row[None, :]
    alpha_quad = Q * alpha_row[None, :]
    mu_quad = Q * mu_row[None, :]

    F = np.vstack((alpha_lin, mu_lin, alpha_quad, mu_quad))

    # Classificazione dei monomi quadratici
    num_q = Q.shape[0]
    is_yy = np.zeros(num_q, dtype=bool)
    is_yz = np.zeros(num_q, dtype=bool)
    is_zz = np.zeros(num_q, dtype=bool)

    row = 0
    for i in range(ry + rz):
        for j in range(i, ry + rz):
            if i < ry and j < ry:
                is_yy[row] = True
            elif i >= ry and j >= ry:
                is_zz[row] = True
            else:
                is_yz[row] = True
            row += 1

    rtot = ry + rz
    idx_alpha_lin = np.arange(0, rtot)
    idx_mu_lin = np.arange(rtot, 2 * rtot)
    idx_alpha_quad = np.arange(2 * rtot, 2 * rtot + num_q)
    idx_mu_quad = np.arange(2 * rtot + num_q, 2 * rtot + 2 * num_q)

    # Maschera dei coefficienti ammessi
    masks = np.zeros((rtot, F.shape[0]), dtype=bool)

    for i in range(rtot):
        if i < ry:
            # Equazioni dello stato originale
            masks[i, idx_alpha_lin[:ry]] = True
            masks[i, idx_mu_lin[:ry]] = True
            masks[i, idx_mu_quad[is_yz]] = True
        else:
            # Equazioni dello stato sollevato
            masks[i, idx_alpha_quad[is_yy]] = True
            masks[i, idx_mu_lin[ry:]] = True
            masks[i, idx_mu_quad[is_zz]] = True

    return Psi_y, Psi_z, sv_y, sv_z, F, X_dot, masks


def OpInf(F, X_dot, masks, reg):
    """Operator Inference con maschera e regolarizzazione Tikhonov."""
    D = F.T
    coefficients = np.zeros((X_dot.shape[0], F.shape[0]), dtype=float)

    for i in range(X_dot.shape[0]):
        active_cols = np.flatnonzero(masks[i, :])
        D_act = D[:, active_cols]

        # Normalizzazione colonne
        col_norms = np.sqrt(np.sum(D_act ** 2, axis=0))
        col_norms[col_norms == 0.0] = 1.0
        D_scaled = D_act / col_norms[None, :]

        # Regolarizzazione
        num_act = D_scaled.shape[1]
        augmented_matrix = np.vstack((
            D_scaled,
            np.sqrt(reg) * np.eye(num_act),
        ))
        augmented_target = np.concatenate((
            X_dot[i, :],
            np.zeros(num_act),
        ))

        # MATLAB backslash su sistema rettangolare -> least squares
        sol, *_ = np.linalg.lstsq(augmented_matrix, augmented_target, rcond=None)
        coefficients[i, active_cols] = sol / col_norms

    return coefficients


def fit_allen_lift_coefficients(F, X_dot, masks, regularization):
    coefficients = OpInf(F, X_dot, masks, regularization)
    training_residual = (
        np.linalg.norm(coefficients @ F - X_dot, ord='fro')
        / np.linalg.norm(X_dot, ord='fro')
    )
    return coefficients, training_residual


def simulate_allen_pod(Psi, Lr, y0, alpha, mu, dt, times, save):
    """ROM POD-Galerkin con Eulero esplicito."""
    a_k = Psi.T @ y0
    ry = Psi.shape[1]

    A = np.zeros((ry, (len(times) - 1) // save + 1), dtype=float)
    A[:, 0] = a_k

    tic = time.perf_counter()
    save_counter = 0

    for step in range(1, len(times)):
        full_state = Psi @ a_k
        da = alpha * (Lr @ a_k) + mu * (a_k - Psi.T @ (full_state ** 3))
        a_k = a_k + dt * da

        if step % save == 0:
            save_counter += 1
            A[:, save_counter] = a_k

    runtime = time.perf_counter() - tic
    reconstructed_state = Psi @ A[:, :save_counter + 1]
    return reconstructed_state, runtime


def simulate_allen_deim(Psi, Lr, proj, s_Psi, y0, alpha, mu, dt, times, save):
    """ROM POD-DEIM con Eulero esplicito."""
    a_k = Psi.T @ y0
    ry = Psi.shape[1]

    A = np.zeros((ry, (len(times) - 1) // save + 1), dtype=float)
    A[:, 0] = a_k

    tic = time.perf_counter()
    save_counter = 0

    for step in range(1, len(times)):
        sampled_state = s_Psi @ a_k
        da = alpha * (Lr @ a_k) + mu * (a_k - proj @ (sampled_state ** 3))
        a_k = a_k + dt * da

        if step % save == 0:
            save_counter += 1
            A[:, save_counter] = a_k

    runtime = time.perf_counter() - tic
    reconstructed_state = Psi @ A[:, :save_counter + 1]
    return reconstructed_state, runtime


def simulate_allen_lift(y0, Psi_y, Psi_z, coefficients, alpha, mu, dt, times, save_stride):
    """Simulazione Lift & Learn, ricostruendo a_z da a_y ad ogni passo."""
    ry = Psi_y.shape[1]
    rz = Psi_z.shape[1]

    # Matrice per ricostruire esattamente il lifting quadratico proiettato
    M = np.zeros((rz, ry * (ry + 1) // 2), dtype=float)
    col = 0

    for i in range(ry):
        for j in range(i, ry):
            prod_vec = Psi_y[:, i] * Psi_y[:, j]
            if i < j:
                prod_vec = 2.0 * prod_vec
            M[:, col] = Psi_z.T @ prod_vec
            col += 1

    a_y = Psi_y.T @ y0
    A = np.zeros((ry, (len(times) - 1) // save_stride + 1), dtype=float)
    A[:, 0] = a_y

    tic = time.perf_counter()
    save_counter = 0

    for step in range(1, len(times)):
        q_y = qvec(a_y)
        a_z = M @ q_y
        state_all = np.concatenate((a_y, a_z))
        quad_all = qvec(state_all)

        features = np.concatenate((
            alpha * state_all,
            mu * state_all,
            alpha * quad_all,
            mu * quad_all,
        ))

        full_deriv = coefficients @ features
        a_y = a_y + dt * full_deriv[:ry]

        if step % save_stride == 0:
            save_counter += 1
            A[:, save_counter] = a_y

    runtime = time.perf_counter() - tic
    solution = Psi_y @ A[:, :save_counter + 1]
    return solution, runtime


## Parametri, griglia spaziale e discretizzazione temporale

In [ ]:

n = 24
T = 2.0
N = n * n  # nodi interni della griglia spaziale

dx = 1.0 / (n + 1)
x = np.linspace(dx, 1.0 - dx, n)

# Costruzione Laplaciano
# scipy.sparse.spdiags replica la costruzione tridiagonale MATLAB.
e = np.ones(n)
L1 = sp.spdiags(
    np.vstack((e, -2.0 * e, e)),
    [-1, 0, 1],
    n,
    n,
).tocsc() / dx**2

L = (
    sp.kron(sp.eye(n, format='csc'), L1, format='csc')
    + sp.kron(L1, sp.eye(n, format='csc'), format='csc')
).tocsc()

# MATLAB: [X1, X2] = meshgrid(x, x)
X1, X2 = np.meshgrid(x, x, indexing='xy')

# MATLAB: reshape((campo)', N, 1), con linearizzazione column-major.
initial_field = 0.1 * np.sin(np.pi * X1) * np.sin(np.pi * X2)
y0 = initial_field.T.reshape(N, order='F')

# Discretizzazione temporale
snapshot_dt = 0.04
dt = 0.0004
num_steps = int(np.ceil(T / dt))
times = np.linspace(0.0, T, num_steps + 1)
save = int(np.rint(snapshot_dt / dt))
snapshot_times = times[::save]

# Parametri di test: non presenti nel training
test_alpha = 0.2
test_mu = 8.0

print(f'N = {N}, dx = {dx:.6f}, passi temporali = {len(times)-1}')
print(f'snapshot ogni {save} passi -> {len(snapshot_times)} snapshot per simulazione')


## Dati di training: generazione snapshot FOM

In [ ]:

alpha_values = np.array([0.01, 0.1, 1.0])
mu_values = np.array([1.0, 6.0, 11.0])

# MATLAB meshgrid + A(:)' / M(:)': flatten in ordine Fortran.
A_grid, M_grid = np.meshgrid(alpha_values, mu_values, indexing='xy')
training_alphas = A_grid.reshape(-1, order='F')
training_mus = M_grid.reshape(-1, order='F')

num_sims = len(training_alphas)  # 9
num_snaps_per_sim = len(snapshot_times)
snapshot_matrix = np.zeros((N, num_snaps_per_sim * num_sims), dtype=float)

fom_train_time = 0.0

for k in range(num_sims):
    states, rt = simulate_allen_fom_implicit(
        L, y0, dt, times, save, snapshot_times,
        training_alphas[k], training_mus[k]
    )

    col_start = k * num_snaps_per_sim
    col_end = (k + 1) * num_snaps_per_sim
    snapshot_matrix[:, col_start:col_end] = states
    fom_train_time += rt

    print(
        f'FOM training {k+1:>2}/{num_sims} | '
        f'alpha={training_alphas[k]:g}, mu={training_mus[k]:g} | '
        f'{rt:.3f} s'
    )

print(f'\nTempo FOM totale per generazione training: {fom_train_time:.3f} s')


## Test FOM

In [ ]:

test_reference_traj_coarse, test_fom_time = simulate_allen_fom_implicit(
    L, y0, dt, times, save, snapshot_times, test_alpha, test_mu
)

print(f'Tempo FOM test: {test_fom_time:.6f} s')


## Addestramento e simulazione: POD e POD-DEIM

In [ ]:

r_pod = 8
m_deim = 10

(
    Psi,
    Lr,
    sv_linear,
    nonlinear_projection,
    sampled_Psi,
    deim_idx,
    sv_nonlinear,
) = train_allen_pod(L, snapshot_matrix, r_pod, m_deim)

# POD e POD-DEIM sui dati di training
training_errors_pod = np.zeros(num_sims)
training_errors_deim = np.zeros(num_sims)

for k in range(num_sims):
    col_start = k * num_snaps_per_sim
    col_end = (k + 1) * num_snaps_per_sim
    fom_train_esatto = snapshot_matrix[:, col_start:col_end]

    pod_tr, _ = simulate_allen_pod(
        Psi, Lr, y0,
        training_alphas[k], training_mus[k],
        dt, times, save
    )
    deim_tr, _ = simulate_allen_deim(
        Psi, Lr, nonlinear_projection, sampled_Psi, y0,
        training_alphas[k], training_mus[k],
        dt, times, save
    )

    training_errors_pod[k] = (
        np.linalg.norm(pod_tr - fom_train_esatto, ord='fro')
        / np.linalg.norm(fom_train_esatto, ord='fro')
    )
    training_errors_deim[k] = (
        np.linalg.norm(deim_tr - fom_train_esatto, ord='fro')
        / np.linalg.norm(fom_train_esatto, ord='fro')
    )

# Test POD e POD-DEIM sul punto non visto
pod_traj, pod_time = simulate_allen_pod(
    Psi, Lr, y0, test_alpha, test_mu, dt, times, save
)
deim_traj, deim_time = simulate_allen_deim(
    Psi, Lr, nonlinear_projection, sampled_Psi,
    y0, test_alpha, test_mu, dt, times, save
)

pod_test_err = (
    np.linalg.norm(pod_traj - test_reference_traj_coarse, ord='fro')
    / np.linalg.norm(test_reference_traj_coarse, ord='fro')
)
deim_test_err = (
    np.linalg.norm(deim_traj - test_reference_traj_coarse, ord='fro')
    / np.linalg.norm(test_reference_traj_coarse, ord='fro')
)

print(f'Errore test POD      = {pod_test_err:.6e}')
print(f'Errore test POD-DEIM = {deim_test_err:.6e}')


## Costruzione basi Lift & Learn

In [ ]:

ry_lift = 4
rz_lift = 4

(
    Psi_y,
    Psi_z,
    sv_y,
    sv_z,
    lift_F,
    lift_Xdot,
    lift_masks,
) = build_allen_lift_data(
    L,
    training_alphas,
    training_mus,
    snapshot_matrix,
    num_snaps_per_sim,
    ry_lift,
    rz_lift,
)


## Energia trattenuta al variare di r — modello Lifted

In [ ]:

r_test_values = np.array([1, 2, 4, 8, 16, 32], dtype=int)
num_tests = len(r_test_values)

total_energy_y = np.sum(sv_y ** 2)
total_energy_z = np.sum(sv_z ** 2)

Energia_Y_perc = np.zeros(num_tests)
Energia_Z_perc = np.zeros(num_tests)

for i, r_curr in enumerate(r_test_values):
    r_curr_y = min(r_curr, len(sv_y))
    r_curr_z = min(r_curr, len(sv_z))

    Energia_Y_perc[i] = (
        np.sum(sv_y[:r_curr_y] ** 2) / total_energy_y
    ) * 100.0
    Energia_Z_perc[i] = (
        np.sum(sv_z[:r_curr_z] ** 2) / total_energy_z
    ) * 100.0

Tabella_EnergiaVsR = pd.DataFrame({
    'Dimensione_r': r_test_values,
    'Energia_Y_perc': Energia_Y_perc,
    'Energia_Z_perc': Energia_Z_perc,
})

print('\nENERGIA TRATTENUTA AL VARIARE DELLA BASE LIFTED\n')
display(Tabella_EnergiaVsR)


## Energia trattenuta e dimensione dei modelli

In [ ]:

en_pod_y = np.sum(sv_linear[:r_pod] ** 2) / np.sum(sv_linear ** 2)
en_deim_f = np.sum(sv_nonlinear[:m_deim] ** 2) / np.sum(sv_nonlinear ** 2)
en_lift_y = np.sum(sv_y[:ry_lift] ** 2) / np.sum(sv_y ** 2)
en_lift_z = np.sum(sv_z[:rz_lift] ** 2) / np.sum(sv_z ** 2)

Metodo_ROM = ['POD Galerkin', 'POD-DEIM', 'Lift & Learn']
Modi_Stato = [r_pod, r_pod, ry_lift]
Modi_NonLineari = [np.nan, m_deim, rz_lift]
Energia_Stato_Perc = [en_pod_y * 100, en_pod_y * 100, en_lift_y * 100]
Energia_NonLineare_Perc = [np.nan, en_deim_f * 100, en_lift_z * 100]

Tabella_ModelliScelti = pd.DataFrame({
    'Metodo_ROM': Metodo_ROM,
    'Modi_Stato': Modi_Stato,
    'Modi_NonLineari': Modi_NonLineari,
    'Energia_Stato_Perc': Energia_Stato_Perc,
    'Energia_NonLineare_Perc': Energia_NonLineare_Perc,
})

print('\nENERGIA TRATTENUTA DAI MODELLI SCELTI\n')
display(Tabella_ModelliScelti)


## Regolarizzazione e simulazione Lift & Learn

In [ ]:

candidate_regularizations = np.array([
    1e-10, 1e-9, 1e-8, 1e-7, 1e-6,
    1e-5, 1e-4, 1e-2, 1.0,
])

best_error = np.inf
best_regularization = np.nan
best_coefficients = None
best_lift_traj = None
best_lift_time = 0.0

for reg in candidate_regularizations:
    coeff_candidate, _ = fit_allen_lift_coefficients(
        lift_F, lift_Xdot, lift_masks, reg
    )

    # Stabilità sul training
    training_stable = True
    for k in range(num_sims):
        train_traj_candidate, _ = simulate_allen_lift(
            y0, Psi_y, Psi_z, coeff_candidate,
            training_alphas[k], training_mus[k],
            dt, times, save
        )

        if np.any(~np.isfinite(train_traj_candidate)):
            training_stable = False
            break

    if not training_stable:
        continue

    # Stabilità sul test
    test_traj_candidate, lift_time_candidate = simulate_allen_lift(
        y0, Psi_y, Psi_z, coeff_candidate,
        test_alpha, test_mu, dt, times, save
    )

    if not np.all(np.isfinite(test_traj_candidate)):
        continue

    # Errore relativo di test
    err_test = (
        np.linalg.norm(test_traj_candidate - test_reference_traj_coarse, ord='fro')
        / np.linalg.norm(test_reference_traj_coarse, ord='fro')
    )

    if err_test < best_error:
        best_error = err_test
        best_regularization = reg
        best_coefficients = coeff_candidate
        best_lift_traj = test_traj_candidate
        best_lift_time = lift_time_candidate

if np.isnan(best_regularization):
    raise RuntimeError('Nessun modello stabile trovato.')

print(
    f'Regolarizzazione = {best_regularization:.1e} '
    f'(Errore Test = {best_error:.3e})'
)

lift_coefficients = best_coefficients
lift_test_err = best_error

# Errori Lift & Learn sui punti di training
training_errors_lift = np.zeros(num_sims)

for k in range(num_sims):
    col_start = k * num_snaps_per_sim
    col_end = (k + 1) * num_snaps_per_sim
    fom_train_esatto = snapshot_matrix[:, col_start:col_end]

    lift_tr, _ = simulate_allen_lift(
        y0, Psi_y, Psi_z, best_coefficients,
        training_alphas[k], training_mus[k],
        dt, times, save
    )

    training_errors_lift[k] = (
        np.linalg.norm(lift_tr - fom_train_esatto, ord='fro')
        / np.linalg.norm(fom_train_esatto, ord='fro')
    )


## Tabelle riassuntive finali

In [ ]:

# Risultati TEST
Modello = ['FOM Esatto', 'POD', 'POD-DEIM', 'Lift & Learn']
Errore_Relativo = np.array([0.0, pod_test_err, deim_test_err, lift_test_err])
Tempo_Secondi = np.array([test_fom_time, pod_time, deim_time, best_lift_time])
Speedup = test_fom_time / Tempo_Secondi

Tabella_test = pd.DataFrame({
    'Modello': Modello,
    'Errore_Relativo': Errore_Relativo,
    'Tempo_Secondi': Tempo_Secondi,
    'Speedup': Speedup,
})

print('\nTABELLA RISULTATI TEST\n')
display(Tabella_test)

# Errori sui punti di training
Tabella_Training = pd.DataFrame({
    'Alpha_Training': training_alphas,
    'Mu_Training': training_mus,
    'Errore_POD': training_errors_pod,
    'Errore_POD_DEIM': training_errors_deim,
    'Errore_Lift_Learn': training_errors_lift,
})

print('\nTABELLA ERRORE SUI PUNTI DI TRAINING\n')
display(Tabella_Training)

# Riepilogo training vs punto non visto
Metodo_Confronto = ['POD Galerkin', 'POD-DEIM', 'Lift & Learn']
Errore_Training_Medio = np.array([
    np.mean(training_errors_pod),
    np.mean(training_errors_deim),
    np.mean(training_errors_lift),
])
Errore_Training_Max = np.array([
    np.max(training_errors_pod),
    np.max(training_errors_deim),
    np.max(training_errors_lift),
])
Errore_Test_NonVisto = np.array([
    pod_test_err,
    deim_test_err,
    lift_test_err,
])

Tabella_RiepilogoTraining = pd.DataFrame({
    'Metodo_Confronto': Metodo_Confronto,
    'Errore_Training_Medio': Errore_Training_Medio,
    'Errore_Training_Max': Errore_Training_Max,
    'Errore_Test_NonVisto': Errore_Test_NonVisto,
})

print('\nRIEPILOGO TRAINING vs PUNTO NON VISTO\n')
display(Tabella_RiepilogoTraining)


## Grafici

In [ ]:

# MATLAB center_i = floor(n/2)+1 (1-based) -> Python center_i = n//2 (0-based)
center_i = n // 2
center_index = center_i * n + center_i
line_styles = ['k-', 'b--', 'g:', 'r-.']

# Evoluzione temporale al centro del dominio
plt.figure(figsize=(9, 4.5))
plt.plot(
    snapshot_times,
    test_reference_traj_coarse[center_index, :],
    line_styles[0],
    label=Modello[0],
    linewidth=1.5,
)
plt.plot(
    snapshot_times,
    pod_traj[center_index, :],
    line_styles[1],
    label=Modello[1],
    linewidth=1.5,
)
plt.plot(
    snapshot_times,
    deim_traj[center_index, :],
    line_styles[2],
    label=Modello[2],
    linewidth=2.0,
)
plt.plot(
    snapshot_times,
    best_lift_traj[center_index, :],
    line_styles[3],
    label=Modello[3],
    linewidth=1.5,
)
plt.xlabel('Tempo [s]')
plt.ylabel('y(0.5, 0.5, t)')
plt.title(rf'Evoluzione Centro Dominio ($\alpha={test_alpha:g}$, $\mu={test_mu:g}$)')
plt.legend(loc='best')
plt.grid(True)
plt.tight_layout()
plt.show()

# FOM esatto e Lift & Learn allo stato finale
fig, axes = plt.subplots(1, 2, figsize=(9, 4.5))

fom_final = test_reference_traj_coarse[:, -1].reshape((n, n), order='F').T
lift_final = best_lift_traj[:, -1].reshape((n, n), order='F').T

im0 = axes[0].imshow(
    fom_final,
    extent=[0, 1, 0, 1],
    origin='lower',
    aspect='equal',
)
axes[0].set_xlabel(r'$x_1$')
axes[0].set_ylabel(r'$x_2$')
axes[0].set_title('Soluzione FOM finale')
fig.colorbar(im0, ax=axes[0])

im1 = axes[1].imshow(
    lift_final,
    extent=[0, 1, 0, 1],
    origin='lower',
    aspect='equal',
)
axes[1].set_xlabel(r'$x_1$')
axes[1].set_ylabel(r'$x_2$')
axes[1].set_title('Ricostruzione Lift & Learn finale')
fig.colorbar(im1, ax=axes[1])

plt.tight_layout()
plt.show()

# Errore assoluto finale Lift & Learn
final_error = np.abs(best_lift_traj[:, -1] - test_reference_traj_coarse[:, -1])
final_error_field = final_error.reshape((n, n), order='F').T

plt.figure(figsize=(6, 5))
im = plt.imshow(
    final_error_field,
    extent=[0, 1, 0, 1],
    origin='lower',
    aspect='equal',
)
plt.colorbar(im)
plt.xlabel(r'$x_1$')
plt.ylabel(r'$x_2$')
plt.title('Errore assoluto finale Lift & Learn')
plt.tight_layout()
plt.show()

# Spettro SVD
sv_y_norm = sv_y / sv_y[0]
sv_z_norm = sv_z / sv_z[0]
num_plot = min(20, len(sv_y_norm), len(sv_z_norm))
mode_idx = np.arange(1, num_plot + 1)

plt.figure(figsize=(8, 4.8))
plt.semilogy(
    mode_idx,
    sv_y_norm[:num_plot],
    'b-',
    label='Valori singolari (blocco y)',
    linewidth=2,
)
plt.semilogy(
    mode_idx,
    sv_z_norm[:num_plot],
    'r--',
    label=r'Valori singolari (blocco z = $y^2$)',
    linewidth=2,
)
plt.xlabel('Indice del modo (i)')
plt.ylabel(r'$\sigma_i / \sigma_1$')
plt.title('Decadimento dei valori singolari')
plt.legend(loc='upper right')
plt.grid(True)
plt.tight_layout()
plt.show()



### Grafico opzionale commentato nel MATLAB originale

Il blocco seguente corrisponde al bar chart degli errori di training che nel file MATLAB era commentato. È lasciato commentato anche qui, così il comportamento del notebook resta fedele all'originale.


In [ ]:

# training_labels = [
#     rf'$\alpha={a:g}$\n$\mu={m:g}$'
#     for a, m in zip(training_alphas, training_mus)
# ]
# bar_data = np.column_stack((
#     training_errors_pod,
#     training_errors_deim,
#     training_errors_lift,
# ))
#
# x_pos = np.arange(num_sims)
# width = 0.25
#
# plt.figure(figsize=(9, 4.8))
# plt.bar(x_pos - width, bar_data[:, 0], width=width, label='POD')
# plt.bar(x_pos,         bar_data[:, 1], width=width, label='POD-DEIM')
# plt.bar(x_pos + width, bar_data[:, 2], width=width, label='Lift & Learn')
# plt.yscale('log')
# plt.xticks(x_pos, training_labels)
# plt.xlabel('Combinazione di addestramento')
# plt.ylabel('Errore relativo (scala log)')
# plt.title('Errore sui dati di training: POD vs POD-DEIM vs Lift & Learn')
# plt.legend(loc='best')
# plt.grid(True)
# plt.tight_layout()
# plt.show()



## Note di equivalenza MATLAB/Python

- `A \ b` → `spsolve(A, b)` per i sistemi sparsi di Newton.
- `A / B` → risoluzione di un sistema trasposto, equivalente a `A @ inv(B)` senza calcolare l'inversa esplicita.
- `svd(X, 'econ')` → `np.linalg.svd(X, full_matrices=False)`.
- `table(...)` → `pandas.DataFrame(...)`.
- `tic/toc` → `time.perf_counter()`.
- `reshape(..., n, n)'` MATLAB → `reshape((n, n), order='F').T` in NumPy.

L'indicizzazione Python è 0-based; tutti gli indici sono stati adattati mantenendo invariata la matematica del codice originale.
